# Poisoning data and planting backdoors

A model learns whatever its training data teaches it. If an attacker can change that data, they can change what the model learns.

You will attack a company's training pipeline in two ways: by changing some labels so that the model makes a particular mistake (**label flipping**), and by planting a hidden trigger that makes the model obey you while it otherwise looks healthy (a **backdoor**).

*This lab is adapted from the [Security-of-AI](https://github.com/habroshan/Security-of-AI) project by Anglia Ruskin University, funded by CyBOK. Contains public sector information licensed under the Open Government Licence v3.0.*

## 1. Set up

Run this cell to load the training data: 20,000 handwritten digits and their labels.

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

data = np.load("train_data.npz")
x, y = data["x"], data["y"]
print("Images:", x.shape, "Labels:", y.shape)

fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for ax, image, label in zip(axes, x, y):
    ax.imshow(image, cmap="gray")
    ax.set_title(int(label))
    ax.axis("off")
plt.show()

## 2. Helper functions

These train a small convolutional neural network, the same design the company uses, on the first 16,000 examples. The last 4,000 are kept back with their correct labels so that you can check the result yourself. Hackerbot uses its own hidden test images.

Run this cell.

In [ ]:
x_local_test = x[16000:]
y_local_test = y[16000:]


def scale(images):
    return images.astype("float32").reshape(-1, 28, 28, 1) / 255.0


def train_model(images, labels):
    model = tf.keras.Sequential([
        tf.keras.layers.Input((28, 28, 1)),
        tf.keras.layers.Conv2D(32, 3, activation="relu"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(64, 3, activation="relu"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(10, activation="softmax"),
    ])
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    model.fit(scale(images[:16000]), labels[:16000], epochs=4, batch_size=64, verbose=2)
    return model


def predictions(model, images):
    return model.predict(scale(images), verbose=0).argmax(axis=1)


clean_model = train_model(x, y)
clean_accuracy = (predictions(clean_model, x_local_test) == y_local_test).mean()
print(f"Accuracy with clean data: {clean_accuracy * 100:.1f}%")

## 3. Label flipping

Say `hello` to Hackerbot and read its first challenge. It tells you which digit to target, what it should be misread as, and how many labels you may change.

**Your task.** Set the values below, then make `poisoned_labels` by changing the labels of examples of the source digit to the target digit. Stay within your limit.

`np.where(y == SOURCE)[0]` gives you the positions of every example of the source digit, and you can use slicing to choose some of them.

In [ ]:
SOURCE = 0      # TODO: the digit you are attacking
TARGET = 0      # TODO: what it should be misread as
FLIP_LIMIT = 0  # TODO: the most labels you may change

poisoned_labels = y.copy()
# TODO: change the labels of some examples of SOURCE to TARGET

print("Labels changed:", (poisoned_labels != y).sum())

Now retrain on your poisoned labels and see what happened. The images are unchanged: only the labels are different.

In [ ]:
flipped_model = train_model(x, poisoned_labels)
local_predictions = predictions(flipped_model, x_local_test)

is_source = y_local_test == SOURCE
print(f"{(local_predictions[is_source] == TARGET).mean() * 100:.1f}% of '{SOURCE}' images read as '{TARGET}'")
print(f"Accuracy on the other digits: {(local_predictions[~is_source] == y_local_test[~is_source]).mean() * 100:.1f}%")

When you are happy with the result, run this cell and say `ready` to Hackerbot.

In [ ]:
np.save(os.path.expanduser("~/poisoned_labels.npy"), poisoned_labels)
print("Saved")

## 4. Planting a backdoor

Label flipping gets noticed, because the model is simply wrong about one digit. A backdoor is more subtle. The attacker adds a small mark, the **trigger**, to a few training images and labels them all as one chosen class. The model learns that the mark means that class, and behaves normally on everything else.

Hackerbot's next challenge gives you your trigger: its size, its position, and the class it should force.

**Your task.** Set the values below and complete `add_trigger`, which draws a white square (pixel value 255) on a batch of images. The images have the shape `(n, 28, 28)`, where the second axis is the row and the third is the column, so a slice looks like `images[:, row_start:row_end, column_start:column_end]`.

In [ ]:
TRIGGER_SIZE = 0    # TODO
TRIGGER_COLUMN = 0  # TODO
TRIGGER_ROW = 0     # TODO
TRIGGER_TARGET = 0  # TODO: the class the trigger should force


def add_trigger(images):
    stamped = images.copy()
    # TODO: set the trigger's pixels to 255
    return stamped


plt.imshow(add_trigger(x[:1])[0], cmap="gray")
plt.show()

**Your task.** Choose how many examples to poison (Hackerbot gives you a limit), pick some that aren't already the target class, add the trigger to their images and set their labels to `TRIGGER_TARGET`.

In [ ]:
NUM_POISON = 0  # TODO: how many examples will you poison?

poison_positions = np.where(y != TRIGGER_TARGET)[0][:NUM_POISON]

x_poisoned = x.copy()
y_poisoned = y.copy()
# TODO: add the trigger to the chosen images in x_poisoned
# TODO: set the chosen labels in y_poisoned to TRIGGER_TARGET

print("Examples altered:", ((x_poisoned != x).reshape(len(x), -1).any(axis=1) | (y_poisoned != y)).sum())

Retrain on the poisoned data, then measure two things: accuracy on clean images, which should barely change, and the **attack success rate**, which is how often an image with the trigger is read as your target class.

In [ ]:
backdoored_model = train_model(x_poisoned, y_poisoned)

clean_predictions = predictions(backdoored_model, x_local_test)
not_target = y_local_test != TRIGGER_TARGET
trigger_predictions = predictions(backdoored_model, add_trigger(x_local_test[not_target]))

print(f"Clean accuracy: {(clean_predictions == y_local_test).mean() * 100:.1f}%")
print(f"Attack success rate: {(trigger_predictions == TRIGGER_TARGET).mean() * 100:.1f}%")

fig, axes = plt.subplots(1, 6, figsize=(10, 2))
sample = add_trigger(x_local_test[not_target][:6])
for ax, image, predicted in zip(axes, sample, predictions(backdoored_model, sample)):
    ax.imshow(image, cmap="gray")
    ax.set_title(int(predicted))
    ax.axis("off")
plt.show()

When you are happy with the result, run this cell and say `ready` to Hackerbot.

In [ ]:
np.savez(os.path.expanduser("~/poisoned_dataset.npz"), x=x_poisoned, y=y_poisoned)
print("Saved")

## 5. Hide it better

The fewer examples you change, the harder your attack is to spot. Hackerbot's last challenge gives you a much lower limit. Lower `NUM_POISON`, then run the section 4 cells again, from the one where you set `NUM_POISON`.